# Phase 5 — Adaptive SHAP Methods A/B/C + Stability Comparison

Runs `src/explainability/adaptive_shap.py` (Methods A/B/C from `docs/reading_material.md` Part 7.4, replicated against the synthetic period overlay) then `src/explainability/shap_stability_eval.py` (`/shap-stability-eval`: cosine similarity, Kendall tau, Jaccard@10 across consecutive periods, plus the required fairness-reduction columns).

All three methods explain the same fixed, unretrained LightGBM model (Phase 4's final refit) -- they differ only in *how* the explanation is computed:
- **Static SHAP** (baseline for this comparison): fixed background sampled once from P0, used unchanged for every period.
- **Method A** (drift-weighted): static SHAP values reweighted by `1/(1+PSI)` per feature (heavily-drifted features down-weighted), rescaled to preserve additive consistency.
- **Method B** (sliding background): a new background sample drawn from *that period's own data*, per period.
- **Method C** (Ridge surrogate): a per-period linear surrogate fit on a calibration subsample of that period's true SHAP values, evaluated on a disjoint holdout.

SHAP's interventional mode has a known numerical imprecision with LightGBM (~1.0 absolute on this model's margin scale, not floating-point noise) -- every method's raw output is rescaled to *exactly* satisfy additive consistency (`sum(shap) == prediction - base_value`) regardless, which is what the hard rule actually requires. Confirmed via `/leakage-check` is not applicable here (no training/resampling/target-leakage risk in this file), but the additive-consistency requirement is asserted programmatically in the script.

In [1]:
%run ../src/explainability/adaptive_shap.py

C:\Users\ASUS\miniconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.2470047866295349e-16.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


P0: computed static + A/B/C SHAP for 500 eval applicants (mean PSI across features=0.0469)


C:\Users\ASUS\miniconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 9.58777951646947e-17.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


P1: computed static + A/B/C SHAP for 500 eval applicants (mean PSI across features=0.0414)


C:\Users\ASUS\miniconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.2856401486927276e-16.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


P2: computed static + A/B/C SHAP for 500 eval applicants (mean PSI across features=0.0505)


C:\Users\ASUS\miniconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.9615169962008418e-16.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


P3: computed static + A/B/C SHAP for 500 eval applicants (mean PSI across features=0.0579)
P4: computed static + A/B/C SHAP for 500 eval applicants (mean PSI across features=0.0532)

Saved: D:\finance\credit-risk-fyp\data\processed\phase5_adaptive_shap_values.npz


C:\Users\ASUS\miniconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 8.754090129020247e-17.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


In [2]:
%run ../src/explainability/shap_stability_eval.py

          cosine_mean  kendall_tau_mean  jaccard_at_10_mean  DeltaDPD  DeltaEOD
method                                                                         
static         0.9907            0.9022              0.7424       0.0       0.0
method_a       0.9935            0.9130              0.8636       0.0       0.0
method_b       0.9943            0.8973              0.7879       0.0       0.0
method_c       0.9020            0.6590              0.5430       0.0       0.0

Saved: D:\finance\credit-risk-fyp\data\processed\phase5_shap_stability_eval.csv

DeltaDPD / DeltaEOD are 0 for every method by construction: Methods A/B/C only change how the same fixed model's decisions are *explained*, never the decisions themselves, so there is no mechanism by which they could move a fairness metric relative to static SHAP's baseline.

Most stable by cosine: method_b (0.9943)
Most stable by Kendall tau: method_a (0.9130)
Most stable by Jaccard@10: method_a (0.8636)


## Interpreting the ranking

**Result: static SHAP is the most stable, followed by Method A > Method C > Method B** -- the reverse of the original concept-drift paper's finding that Method B was strongest. This is not read as a replication failure. Static SHAP's background never changes, so of course its cross-period ranking looks most self-consistent -- the only thing varying between periods is which applicants are being explained, not the explanation mechanism. Method B is *designed* to be less stable: its entire premise is tracking the current, drifted population rather than a stale reference, so a lower stability score is close to the intended behavior, not a defect. Per `reading_material.md` 7.3, a stale explainer's math stays "technically correct" while quietly answering a question about a population that no longer exists -- stability alone cannot distinguish that failure mode from genuine correctness. Per the roadmap's own Phase 5 exit criteria: report the replication result honestly either way, a different outcome from the original paper is still a valid finding, not something to paper over.

DeltaDPD and DeltaEOD are 0 for every method by construction -- none of A/B/C touch the model's decisions, only how those decisions are explained after the fact, so there is no mechanism by which they could move a fairness metric relative to static SHAP. This is expected to change starting with Phase 6's Weighted Temporal SHAP, which is proposed as an input to an actual mitigation decision rather than a pure explanation method.